# Diff-FEA + PINN vs MMC: Master Narrative

This notebook is the canonical reproducible story for the thesis: from geometry definitions and load cases, through data generation, to the comparative analyses of differentiable FEA + PINN surrogates versus MMC optimization.



## Story Outline

1. **Environment & Utilities**  
   - Import Plotly, pandas, numpy, torch helpers.  
   - Reuse shared plotting utilities (geometry drawing, styling) that will also feed the PINN/MMC notebooks.
2. **Geometry + Boundary Conditions**  
   - Parse `data/cad/lbracket.json` & `lbracket.vtk`.  
   - Render 2D schematic (matplotlib) + Plotly 3D surface/mesh.  
   - Highlight fixed supports + void region using the same parameters as `Visualise_LBracket_Setup.py`.
3. **Load Case Definitions**  
   - Load `data/results/ansys_payloads.json` and `src/tools/Generate_Helper_Figures.py` helpers.  
   - Interactive toggles for horizontal vs vertical load arrows, and link to MMC/PINN case studies.
4. **Dataset + Experiment Matrix**  
   - Summaries from `data/results/dataset_manifest.md`, `data/results/method_comparison.csv`, MMC manifests.  
   - Visualize coverage histograms that previously lived in static PNGs.
5. **Diff-FEA & PINN Benchmarks**  
   - Recreate Proof_Speedup + distribution figures with inline code (no pre-rendered PNGs).  
   - Pull training/inference timings from `lbracket_diff_fea_log.csv` and `train_multi_geom.py` outputs.
6. **MMC Optimization Summary**  
   - High-level results referencing `mmc_lbracket_log.csv` & vertical load logs.  
   - Provide entry points to the dedicated MMC notebook.
7. **Unified Comparison & Takeaways**  
   - Combine runtime/compliance charts (`Generate_Unified_Comparison.py` logic) with annotations.  
   - List reproducibility steps + next experiments.



## Key Terminology

Before diving into the visualizations, here are key concepts:

- **Compliance**: A measure of structural flexibility (units: Joules). Lower compliance = stiffer structure = better design. $\text{Compliance} = F^T \cdot u$ (force vector dot displacement vector).
- **Optimization**: Finding the best screw positions to minimize compliance (maximize stiffness).
- **FEA (Finite Element Analysis)**: Numerical method to solve structural mechanics equations. Computes displacements and compliance for given geometry/loads.
- **PINN (Physics-Informed Neural Network)**: Neural network trained on FEA data to predict compliance instantly (surrogate model).
- **MMC (Moving Morphable Components)**: Optimization method that moves "components" (screws) through the design space to find optimal positions.
- **Load Case**: Specific force application scenario (e.g., horizontal tip load = 1000N force applied horizontally at the tip).
- **Fixed Support**: Boundary condition where displacement is zero (clamped edge).

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.express as px
import plotly.io as pio

_possible_dirs = [Path.cwd().resolve()]
if (Path.cwd() / 'docs' / 'notebooks').exists():
    _possible_dirs.append(Path.cwd() / 'docs' / 'notebooks')
helper_dir = None
for candidate in _possible_dirs:
    candidate = candidate.resolve()
    if (candidate / 'story_helpers.py').exists():
        helper_dir = candidate
        break
if helper_dir is None:
    raise RuntimeError('Cannot locate story_helpers.py')
if str(helper_dir) not in sys.path:
    sys.path.append(str(helper_dir))

from story_helpers import (
    ROOT,
    RESULTS_DIR,
    assemble_multi_geom_dataframe,
    draw_lbracket_2d,
    list_multi_geom_paths,
    load_ansys_layouts,
    make_bracket_mesh,
    make_geometry_3d,
    make_geometry_with_loads_3d,
    make_ribbed_channel_mesh,
    make_tapered_plate_mesh,
    measure_speed_benchmarks,
    plot_dataset_histogram,
    plot_dataset_scatter,
    plot_load_layouts,
    plot_method_comparison,
    plot_speed_bars,
    plot_unified_convergence,
    summarize_dataset,
    load_csv,
)

pio.renderers.default = "notebook_connected"

# Reload module to pick up changes
import importlib
import story_helpers
importlib.reload(story_helpers)
from story_helpers import (
    ROOT,
    RESULTS_DIR,
    assemble_multi_geom_dataframe,
    draw_lbracket_2d,
    list_multi_geom_paths,
    load_ansys_layouts,
    make_bracket_mesh,
    make_geometry_3d,
    make_geometry_with_loads_3d,
    make_ribbed_channel_mesh,
    make_tapered_plate_mesh,
    measure_speed_benchmarks,
    plot_dataset_histogram,
    plot_dataset_scatter,
    plot_load_layouts,
    plot_method_comparison,
    plot_speed_bars,
    plot_unified_convergence,
    summarize_dataset,
    load_csv,
)

## Geometry & Boundary Conditions

The following cells recreate the CAD visualization directly from the thesis specifications. We draw the planform with Plotly, then extrude it into 3D to highlight thickness and void regions.



In [2]:
# L-Bracket 2D and 3D
geom2d_fig = draw_lbracket_2d()
geom2d_fig.show()

mesh_fig = make_bracket_mesh()
mesh_fig.show()

# Ribbed Channel 3D
ribbed_fig = make_ribbed_channel_mesh()
ribbed_fig.show()

# Tapered Plate 3D
tapered_fig = make_tapered_plate_mesh()
tapered_fig.show()

**What this shows:**
- **2D Planform**: The L-bracket outline showing the aluminum part (gray), void region (dashed rectangle), and fixed support (red line at top of vertical leg)
- **3D Geometry**: Extruded view showing the 5mm thickness. The two legs (vertical and horizontal) are visible as separate colored blocks
- **Purpose**: Establishes the benchmark geometry used throughout the thesis for both PINN and MMC approaches

## Load Cases in 3D

Interactive 3D visualizations showing force vectors applied to each geometry. Use the dropdown to switch between geometries and load cases.

**What this shows:**
- **3D Force Vectors**: Blue/green arrows showing the direction and magnitude of applied loads
- **Fixed Supports**: Red lines indicating where the geometry is clamped (zero displacement boundary conditions)
- **Load Cases**:
  - **L-Bracket**: Horizontal tip load (1000N leftward) or vertical tip load (1000N upward)
  - **Ribbed Channel**: Upward tip load (1000N upward) or lateral shear (1000N downward)
  - **Tapered Plate**: Combined tip load (450N downward force + 25Nm torsion)
- **Purpose**: Visualizes the boundary conditions and loading scenarios that define each optimization problem

In [ ]:
# Interactive 3D load case viewer
# Use the dropdown widgets below to select geometry and load case
# This is much faster than rendering all figures at once

def show_load_case_3d(geometry="l_bracket", load_case="horizontal_tip"):
    """Display a single 3D geometry with load vectors."""
    try:
        fig = make_geometry_with_loads_3d(geometry, load_case)
        # Return figure - Jupyter will auto-display it (no need for fig.show())
        return fig
    except Exception as e:
        print(f"Error rendering {geometry} with {load_case}: {e}")
        return None

# Interactive widget with geometry-dependent load cases
def update_load_options(geometry):
    """Update load case options based on selected geometry."""
    if geometry == "l_bracket":
        return ["horizontal_tip", "vertical_tip"]
    elif geometry == "ribbed_channel":
        return ["upward_tip", "lateral_shear"]
    elif geometry == "tapered_plate":
        return ["combined_tip"]
    return []

geo_widget = widgets.Dropdown(
    options=["l_bracket", "ribbed_channel", "tapered_plate"],
    value="l_bracket",
    description="Geometry:",
    style={"description_width": "initial"}
)

load_widget = widgets.Dropdown(
    options=["horizontal_tip", "vertical_tip"],
    value="horizontal_tip",
    description="Load Case:",
    style={"description_width": "initial"}
)

def on_geometry_change(change):
    """Update load case dropdown when geometry changes."""
    if change["name"] == "value":
        new_options = update_load_options(change["new"])
        load_widget.options = new_options
        if new_options:
            load_widget.value = new_options[0]

geo_widget.observe(on_geometry_change, names="value")

def show_interactive(geometry, load_case):
    """Display the selected geometry and load case."""
    return show_load_case_3d(geometry, load_case)

print("Select geometry and load case from the dropdowns below:")
widgets.interact(show_interactive, geometry=geo_widget, load_case=load_widget);

# Optional: Uncomment below to see all load cases at once (may be slow)
# print("\nShowing all load cases (this may take a moment)...")
# show_load_case_3d("l_bracket", "horizontal_tip")
# show_load_case_3d("l_bracket", "vertical_tip")
# show_load_case_3d("ribbed_channel", "upward_tip")
# show_load_case_3d("ribbed_channel", "lateral_shear")
# show_load_case_3d("tapered_plate", "combined_tip")

Select geometry and load case from the dropdowns below:


interactive(children=(Dropdown(description='Geometry:', options=('l_bracket', 'ribbed_channel', 'tapered_plate…

## Load Cases, Layouts, and Screw Placements

Layouts are sourced from `data/results/ansys_payloads.json`. Instead of static PNGs, we rebuild the screw-placement overview and provide an inspector widget for per-layout metadata.



In [4]:
layouts = load_ansys_layouts()
layout_fig = plot_load_layouts(layouts)
layout_fig.show()

layout_options = {f"{entry['method']} — {entry['tag']}": entry for entry in layouts}

def describe_layout(choice):
    entry = layout_options[choice]
    display(pd.DataFrame(entry).drop(columns=["screws"]) if "screws" in entry else pd.DataFrame(entry))
    screws = entry.get("screws", [])
    if screws:
        screw_df = pd.DataFrame(screws)
        display(screw_df)

widgets.interact(describe_layout, choice=list(layout_options.keys()));


interactive(children=(Dropdown(description='choice', options=('diff_fea — lbracket', 'mmc — lbracket', 'mmc — …

**What this shows:**
- **Optimal Screw Placements**: Blue markers (S1, S2) show where each optimization method placed screws
- **Comparison**: Side-by-side view of Diff-FEA, MMC horizontal, and MMC vertical results
- **Compliance Values**: Each subplot shows the final compliance achieved by that method
- **Purpose**: Compares the optimal solutions found by different optimization approaches on the same L-bracket geometry

## Dataset & Experiment Matrix

We profile both the legacy single-geometry dataset and the refreshed multi-geometry corpus. Each visualization is regenerated in-code to replace prior PNG exports.



In [5]:
legacy_path = RESULTS_DIR / "pinn_training_data.csv"
legacy_summary = summarize_dataset(legacy_path, "Legacy L-bracket (90 samples)")

multi_df = assemble_multi_geom_dataframe()
multi_df_sorted = multi_df.sort_values("samples", ascending=False)

summary_table = pd.DataFrame([
    legacy_summary.__dict__,
]).assign(path=lambda df: df["path"].astype(str))

print("Legacy dataset summary:")
display(summary_table)

print("\nMulti-geometry manifests:")
display(multi_df_sorted)



Legacy dataset summary:


,label,path,samples,compliance_min,compliance_max,compliance_mean
0,Legacy L-bracket (90 samples),/home/dio/thesis_project/data/results/pinn_tra...,90,544.599353,1635.712436,1075.074554



Multi-geometry manifests:


,dataset,samples,compliance_min,compliance_max,compliance_mean,path
2,ribbed channel shear,220,0.619838,0.856086,0.741916,data/results/multi_geom_training/ribbed_channe...
3,ribbed channel upward,220,32.139060,32.438080,32.290524,data/results/multi_geom_training/ribbed_channe...
0,l bracket horizontal,200,205.316302,479.239144,378.568877,data/results/multi_geom_training/l_bracket_hor...
1,l bracket vertical,200,972.439491,1701.221319,1247.291741,data/results/multi_geom_training/l_bracket_ver...
4,tapered plate combined,180,22.419019,23.574407,22.897830,data/results/multi_geom_training/tapered_plate...


**What this shows:**
- **Legacy Dataset**: 90 samples from single-geometry L-bracket training (used in initial PINN experiments)
- **Multi-Geometry Dataset**: Breakdown by geometry showing sample counts, compliance ranges, and means
- **Compliance Range**: Min/max/mean values indicate the spread of structural stiffness across different screw placements
- **Purpose**: Documents the training data corpus and shows how the multi-geometry dataset expands coverage beyond the original L-bracket-only approach

In [6]:
legacy_hist = plot_dataset_histogram(legacy_path, "Legacy PINN dataset")
legacy_hist.show()
legacy_scatter = plot_dataset_scatter(legacy_path, "Legacy PINN dataset")
legacy_scatter.show()

multi_combined_path = RESULTS_DIR / "multi_geom_training"
combined_df = pd.concat([load_csv(p) for p in list_multi_geom_paths()], ignore_index=True)
combined_hist = px.histogram(
    combined_df,
    x="compliance",
    color="geometry",
    nbins=30,
    opacity=0.75,
    title="Multi-Geometry Compliance Distribution",
)
combined_hist.update_layout(xaxis_title="Compliance (J)", yaxis_title="Frequency")
combined_hist.show()

multi_scatter = px.scatter(
    combined_df,
    x="s1_x",
    y="s1_y",
    color="geometry",
    hover_data=["load_case", "compliance"],
    title="Primary Screw Position Coverage (Multi-Geometry)",
)
multi_scatter.update_layout(xaxis=dict(scaleanchor="y", scaleratio=1), xaxis_title="X (mm)", yaxis_title="Y (mm)")
multi_scatter.show()



**What this shows:**
- **Compliance Histogram**: Distribution of compliance values (Joules) across all training samples. Lower compliance = stiffer structure
- **Spatial Scatter Plot**: Screw positions colored by compliance. Red/yellow = high compliance (flexible), blue/purple = low compliance (stiff)
- **Multi-Geometry Coverage**: Shows how different geometries contribute to the dataset and their compliance ranges
- **Purpose**: 
  - Histogram: Shows data distribution and whether we have good coverage of the design space
  - Scatter: Reveals spatial patterns (e.g., do optimal screws cluster in high-stress regions?)
  - Multi-geometry: Demonstrates dataset diversity across different part geometries

## Diff-FEA + PINN Benchmarks vs MMC

This section rebuilds the historical PNGs (`Proof_Speedup`, unified comparisons, compliance bar charts) with Plotly and live data.



In [7]:
speed_df = measure_speed_benchmarks()
speed_fig = plot_speed_bars(speed_df)
speed_fig.show()

convergence_fig = plot_unified_convergence()
convergence_fig.show()

compliance_fig = plot_method_comparison()
compliance_fig.show()



**What this shows:**
- **Visual Aid**: This figure displays final compliance values for the L-bracket optimization as a visual reference
- **L-Bracket Only**: Results shown are specifically for the L-bracket geometry with horizontal tip load
- **Dual Y-Axes**: 
  - **Left axis (blue)**: Diff-FEA/PINN results - shows actual compliance values in Joules (~940 J)
  - **Right axis (orange)**: MMC results - shows normalized compliance values (~0.426)
- **Note on Units**: The different units (Joules vs. normalized) prevent direct numerical comparison, but both methods successfully optimized the L-bracket design
- **Lower is Better**: Compliance measures structural flexibility - lower values mean stiffer (better) designs
- **Purpose**: Serves as a visual reference showing that both methods achieved optimized solutions, with actual values displayed as annotations

**What this shows:**
- **Runtime Comparison**: Bar chart (log scale) comparing time per iteration for each method
- **Methods**:
  - **Diff-FEA (Julia)**: ~10.4ms - Full finite element analysis with automatic differentiation
  - **PINN Surrogate**: ~0.009ms - Neural network inference (1000× faster than FEA)
  - **MMC (Python)**: ~118ms - Moving morphable components optimization with FEA solve
- **Speedup Factor**: Shows how much faster PINN inference is compared to traditional FEA
- **Purpose**: Quantifies the computational advantage of using a trained surrogate model vs. solving FEA at each optimization step

In [8]:
benchmark_summary = speed_df.copy()
benchmark_summary["time_ms"] = benchmark_summary["time_s"] * 1000
benchmark_summary["relative_to_fea"] = benchmark_summary["time_s"] / benchmark_summary.loc[
    benchmark_summary["method"] == "Diff-FEA (Julia)", "time_s"
].values[0]
display(benchmark_summary)



,method,time_s,time_ms,relative_to_fea
0,Diff-FEA (Julia),0.010359,10.359440,1.000000
1,PINN Surrogate,0.000105,0.105140,0.010149
2,MMC (Python),0.096609,96.609116,9.325708


## Next Steps & Deep Dives

- Use `pinn_story.ipynb` for the full differentiable FEA + PINN walkthrough (data generation, training curves, generalization surfaces).  
- Use `mmc_story.ipynb` for MMC-specific derivations, parameter sweeps, and 3D screw trajectory reconstructions.  
- Rerun this notebook after updating any CSV/log artifacts to refresh all thesis figures automatically.

